## Run Postprocessing

In [12]:
from rupsycho.parsers.judges import MultipleChoiceJudge, ModelBasedAnswerJudge
from rupsycho.parsers.validators import ApologiesValidatorParser, BeingAiValidatorParser, RefusalValidatorParser, ValidatorParser, ModelBasedValidator
from rupsycho.parsers.cleaners import RegexExtractorCleaner
from rupsycho.postprocessing import PostprocessingPipeline


import rupsycho as rup
from tqdm import tqdm
tqdm.pandas()

## Paths to results to process

In [13]:
config_file_path = "./data/experiments/Exp5/final/bdi_reversed_chatgpt_mini.json"
results_file_patterns = ["./data/experiments/Exp5/final/output/bdi_reversed_chatgpt_mini*.csv"]

## Define Parsers

**Cleaner:**

In [14]:
pattern = r'"answer":\s*"?([^"]*?)"?\s*}'
cleaner = RegexExtractorCleaner(pattern)
cleaner.invoke("answer: I always do my best to be honest.")

'answer: I always do my best to be honest.'

**Validator:**

In [15]:
validator = ValidatorParser()
validator.invoke("I always do my best to be honest.")

{'text': 'I always do my best to be honest.',
 'validation_status': 'valid',
 'details': {'apologies': False, 'being_ai': False, 'refusal': False}}

In [16]:
validator.invoke("I am sorry, I always do my best to be honest.")

{'text': 'I am sorry, I always do my best to be honest.',
 'validation_status': 'invalid',
 'details': {'apologies': True, 'being_ai': False, 'refusal': False}}

**Judge:**

In [17]:
judge = MultipleChoiceJudge(possible_answers=["1. Strongly disagree", "2. Disagree", "3. Somewhat disagree", "4. Neither agree nor disagree", "5. Somewhat agree", "6. Agree", "7. Strongly agree"], ignore_case=False)

In [18]:
cleaned_answer = "Somewhat agree"
judge.invoke(cleaned_answer)

'5. Somewhat agree'

## Define Pipeline

In [19]:
pipeline = PostprocessingPipeline(
	config_file_path, results_file_patterns, # Input files
	cleaner, validator, judge,  # Parser instances
 	output_path="./processed_results.csv" # Output file						  
)

In [20]:
# answer_options = pipeline.experiment.questionnaire.instruction_items[0].get_answer_options_as_list()

# print(answer_options)
# print(type(answer_options))

#pipeline.experiment.questionnaire.default_answer_options.get_answer_options_as_list()

## Run Pipeline

In [21]:
pipeline._load_csv_files().columns

Index(['experiment_name', 'instruction_item_id', 'instruction_item',
       'model_id', 'profile_id', 'random_seed', 'answer'],
      dtype='object')

In [22]:
pipeline.run()

Loading data...
Processing data...


100%|██████████| 5250/5250 [00:00<00:00, 19980.74it/s]


Processed results saved to ./processed_results.csv


## Save Results